In [9]:
import pandas as pd
import cassiopeia as cas
import os
import cassiopeia
import gzip
import sys
import re
import numpy as np
import sys, threading
sys.setrecursionlimit(10**7) # max depth of recursion
threading.stack_size(2**27)
import stringdist
import Levenshtein
from collections import Counter

from Levenshtein import distance as lv
# from tqdm.auto import tqdm
import pickle
from multiprocessing import Pool
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

from cassiopeia.preprocess import (
    alignment_utilities,
    constants,
    map_utils,
    doublet_utils,
    lineage_utils,
    UMI_utils,
    utilities,
)

## 0. resolve and error correct
("*_unmapped_filtered_corrected_sorted.collapsed.txt" into one by collapsing sequences per UMI group.
max_mismatch=3)


In [3]:
def resolve_top1_umi_sequence(
    molecule_table: pd.DataFrame,
    output_directory: str,
    sample_name:str,
    min_umi_per_cell: int = 2,
    min_avg_reads_per_umi: float = 2.0,
    min_reads_per_umi: float = 10.0,
    plot: bool = True,
) -> pd.DataFrame:
    
    mt_filter = {} # True will be filtered
    total_numReads = {}
    top_reads = {}
    second_reads = {}
    # first_reads = {}
    
    unique_pairs = molecule_table.groupby(["cellBC", "UMI"], sort=False)
    
    for _, group in tqdm(
        unique_pairs,
        total=len(unique_pairs.size()),
        desc="Resolving UMI sequences",
    ):
    
        # base case - only one sequence
        if group.shape[0] == 1:
            good_readName = group["readName"].iloc[0]
            mt_filter[good_readName] = False
            total_numReads[good_readName] = group["readCount"].values[0]
            top_reads[good_readName] = group["readCount"].values[0]
    
        # more commonly - many sequences for a given UMI
        else:
            group_sort = group.sort_values(
                "readCount", ascending=False, ignore_index=True
            )
            good_readName = group_sort["readName"].iloc[0]
    
            # keep the first entry (highest readCount)
            mt_filter[good_readName] = False
    
            total_numReads[good_readName] = group_sort["readCount"].sum()
            top_reads[good_readName] = group_sort["readCount"].iloc[0]
            second_reads[good_readName] = group_sort["readCount"].iloc[1]
            # first_reads[good_readName] = group_sort["readCount"].iloc[0]
    
            # mark remaining UMIs for filtering
            for i in range(1, group.shape[0]):
                bad_readName = group_sort["readName"].iloc[i]
                mt_filter[bad_readName] = True
    
        # add number of reads to the table
    molecule_table["total_numReads"] = molecule_table["readName"].map(total_numReads)
    molecule_table["top_reads"] = molecule_table["readName"].map(top_reads)
    molecule_table["second_reads"] = molecule_table["readName"].map(second_reads)


    # apply the filter using the hash table created above
    molecule_table["filter"] = molecule_table["readName"].map(mt_filter)
    n_filtered = molecule_table[molecule_table["filter"]].shape[0]
    molecule_table=molecule_table[molecule_table['top_reads']<=10000]

    # logger.info(f"Filtered out {n_filtered} reads.")

    # filter based on status & reindex
    filt_molecule_table = molecule_table[~molecule_table["filter"]].copy()
    filt_molecule_table.drop(columns=["filter"], inplace=True)

    filt_molecule_table = utilities.filter_cells(
        filt_molecule_table,
        min_umi_per_cell=min_umi_per_cell,
        min_avg_reads_per_umi=min_avg_reads_per_umi,
    )
    
    filt_molecule_table=filt_molecule_table[filt_molecule_table['top_reads']>=min_reads_per_umi]
    
    if plot:
        # pp=PdfPages(output_directory+myMouse+"_"+sample_name+"_second_vs_top_reads_frequency_pickSeq_10reads.pdf")
        molecule_table=molecule_table.fillna(0)
        filt_molecule_table=filt_molecule_table.fillna(0)
        # filt_molecule_table=filt_molecule_table[filt_molecule_table['total_numReads']<=10000]


        fig, axes = plt.subplots(1, 1, figsize=(4, 4))#, sharey=True
        fig.suptitle(mySample+' Top_reads VS. second_reads')
        # before
        # sns.scatterplot(ax=axes[0], data=molecule_table, x="top_reads", y="second_reads",color="tab:blue",alpha=0.02,s=5)
        # axes[0].axline([0, 0], slope=1,linestyle='--',color="tab:green")
        # axes[0].set_title("Before filtering")
        # after filt_molecule_table['top_reads']>=min_reads_per_umi
        sns.scatterplot(data=filt_molecule_table,x="top_reads",y="second_reads",color='tab:blue',alpha=0.02,s=5).set_title(mySample+" top_reads vs. second_reads")
        # ax.axline([0, 0], slope=1,linestyle='--',color="tab:orange")
        plt.axline([0, 0], slope=1,linestyle='--',color="tab:orange")
        plt.axline([0, 0], slope=0.5,linestyle='--',color="tab:green") 
        fig.tight_layout()
        # pp.savefig(plt.gcf())
        plt.savefig(
            os.path.join(output_directory, sample_name+"_second_vs_top_reads_pickSeq.png")
        )
        plt.close()

        filt_molecule_table['top_frequency'] = filt_molecule_table.top_reads/filt_molecule_table.total_numReads
        filt_molecule_table['second_frequency'] = filt_molecule_table.second_reads/filt_molecule_table.total_numReads
        ## 2nd plot
        p=sns.jointplot(data=filt_molecule_table,x="top_frequency",y="second_frequency",color='tab:purple',alpha=0.02,s=5,
                        height=5,ratio=3,marginal_ticks=True,marginal_kws=dict(bins=20))
        p.fig.suptitle(mySample+" top_frequency vs. frequency")
        p.fig.tight_layout()
        p.fig.subplots_adjust(top=0.95)
        # pp.savefig(plt.gcf())
        # pp.close()
        plt.savefig(
            os.path.join(output_directory, mySample+"_second_vs_top_reads_frequency_pickSeq.png")
        )
        plt.close()


    return filt_molecule_table

## 1. C4007

In [3]:
myMouse="C4007"
samplesAmplicon = ['LLT-1-old','LLT-2-old','Abdo-old','LLT-1','LLT-2','Abdo','Chest']#'Chest-old',
output_directory="/syn2/zhaolian/3.JiLab/results/1.BarcodeSeq/03.cassiopeia_out/"+myMouse+"/"
min_umi_per_cell=2
min_avg_reads_per_umi=2
min_reads_per_umi=10
for mySample in samplesAmplicon:
    molecule_table=pd.read_csv(output_directory+mySample+'_unmapped_filtered_corrected_sorted.collapsed.txt',header=0,sep='\t')
    print(mySample,'collapesed', str(len(molecule_table['cellBC'].unique())),str(molecule_table['readCount'].sum()))
    ##---------------------------------------------------------------------------
    ## 1. resolve um-----pick top seq
    df=resolve_top1_umi_sequence(molecule_table,output_directory=output_directory,
                                 sample_name=mySample,
                                 min_umi_per_cell=min_umi_per_cell,
                                 min_avg_reads_per_umi=min_avg_reads_per_umi,
                                 min_reads_per_umi=min_reads_per_umi)
    # df.to_csv(output_directory + '/'+mySample+'_resolve_top1_umi_reads10.txt',sep='\t')
    print(mySample, 'resolve_UMI',str(len(df['cellBC'].unique())),str(df['readCount'].sum()))
    
    umi_table=df.copy()
    umi_table=umi_table[umi_table["top_reads"]>=umi_table["second_reads"]*2]
    print(mySample, 'resolve_umi_top_greater_2second',str(len(umi_table['cellBC'].unique())),str(umi_table['top_reads'].sum()))
    # umi_table
    ##---------------------------------------------------------------------------
    ## 2. error correct UMIs
    umi_table['intBC'] =  mySample 
    n_threads=10
    umi_table = cas.pp.error_correct_umis(
        umi_table,
        max_umi_distance=1,
        allow_allele_conflicts=False,
        n_threads=n_threads,
    )
    print(mySample, 'error_correct_umi',str(len(umi_table['cellBC'].unique())),str(umi_table['top_reads'].sum()))
    umi_table.to_csv(output_directory + '/'+mySample+'_umi_table_filtered.csv',sep='\t')
    


LLT-1-old collapesed 499839 55478785


Resolving UMI sequences:   0%|          | 0/3929324 [00:00<?, ?it/s]

[2024-08-26 11:08:25,186]    INFO [main] Filtered out 415966 cells with too few UMIs or too few average number of reads per UMI.
[2024-08-26 11:08:25,493]    INFO [main] Filtered out 1618684 UMIs as a result.
[2024-08-26 11:08:35,680]    INFO [error_correct_umis] Starting...


LLT-1-old resolve_UMI 48027 33650959
LLT-1-old resolve_umi_top_greater_2second 47395 33388761.0


Error-correcting UMIs: 100%|##########| 47395/47395 [01:03<00:00, 747.53it/s]
[2024-08-26 11:10:08,318]    INFO [error_correct_umis] 8239 UMIs Corrected of 164540(5.007000000000001%)
[2024-08-26 11:10:11,293]    INFO [error_correct_umis] Finished in 95.61342787742615 s.


...

Resolving UMI sequences:   0%|          | 0/850874 [00:00<?, ?it/s]

[2024-08-26 12:03:24,388]    INFO [main] Filtered out 165184 cells with too few UMIs or too few average number of reads per UMI.
[2024-08-26 12:03:24,440]    INFO [main] Filtered out 278980 UMIs as a result.
[2024-08-26 12:03:27,573]    INFO [error_correct_umis] Starting...


Chest resolve_UMI 28396 6701706
Chest resolve_umi_top_greater_2second 28173 6599824.0


Error-correcting UMIs: 100%|##########| 28173/28173 [00:26<00:00, 1073.11it/s]
[2024-08-26 12:04:07,456]    INFO [error_correct_umis] 3350 UMIs Corrected of 109973(3.0460000000000003%)
[2024-08-26 12:04:09,298]    INFO [error_correct_umis] Finished in 41.72496795654297 s.


Chest error_correct_umi 28173 6374016.0
